![Week 3 Day 1](../assets/banners/w3d1.png)

# Week 3 Day 1 - Agents: Workflows, Patterns and the ReAct Loop

Welcome to week 3! 🎉

Look at how far GradeWise has come. In **week 1** it learned to read Asha's exam pack and turn it into one clean `ExamPaper` JSON. In **week 2** it learned the CS201 course material: PDFs chunked, embedded and searchable, with retrieval quality *measured*, not guessed.

But so far, every piece has been a straight line: input in, one model call, output out. When the marks in the extracted exam didn't add up, all GradeWise could do was print a warning. This week, GradeWise learns to **check its own work, fix its mistakes and ask a human before it saves anything**. That's the jump from a *chain* to an *agentic system*, and the tool we'll use is **LangGraph**.

Today is the map for the whole week: what an "agent" really is, the five patterns that cover almost every real system, the ReAct loop that powers tool-using agents, and how to design tools a model can actually use well.

![Your 6-week roadmap](../assets/roadmap/week3.png)

### 🎯 By the end of this lab you will
1. 🗺️ Know what's coming this week, and where it plugs into GradeWise
2. 🧠 Explain the difference between a **workflow** and an **agent**, and when to choose each
3. 🧩 Recognise the five agentic patterns: prompt chaining, routing, parallelisation, orchestrator-workers and evaluator-optimiser
4. 🔁 Run the **ReAct loop** by hand, step by step
5. 🛠️ Design tools that a model can't misuse

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | Day 1 - Week 3 Kickoff: Recap and Roadmap | 🎬 |
| 2 | Day 1 - What Is an Agent? Workflows vs Agents and the Core Patterns | 🎬 |
| 3 | Day 1 - The ReAct Loop and Designing Great Tools | 🎬 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Good news for your wallet: <b>almost everything this week runs offline, for free</b>. LangGraph doesn't need an LLM to run a graph, so we teach graphs with plain Python functions. Cells that need a real model are marked <b>🔑</b>. Without an API key they'll show a friendly "missing key" error, and that's fine: skip them and carry on.</span>
        </td>
    </tr>
</table>

## Imports and environment
One cell for everything, as always.

In [ ]:
import json
import os
import re
import statistics
from concurrent.futures import ThreadPoolExecutor

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

from gradewise.config import DATA_DIR
from gradewise.schemas import ExamPaper

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("ℹ️ OpenAI API Key not set. That's OK: only the cells marked 🔑 need a key, the rest run offline.")
    print("   To set one up, see Part 5 of the setup guide (and remember to SAVE .env!)")

# 🔁 RECAP: the exam you extracted in week 1. We load the reference copy so everyone sees the same data.
exam = ExamPaper.model_validate_json(
    (DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8")
)
QUESTIONS = {q.id: q for q in exam.questions}
print(f"Loaded {exam.title}: {len(exam.questions)} questions, {exam.total_marks:g} marks")

## Part 1: Week 3 kickoff: recap and roadmap

🔁 **Where we are**
- **Week 1:** exam pack → `ExamPaper` JSON with structured output, plus a simple validate-and-retry loop in Python
- **Week 2:** course PDFs → a Chroma knowledge base with hybrid search, measured with MRR and nDCG

**This week's milestone** is *exam understanding v2*: the week 1 extractor rebuilt as a LangGraph workflow that checks itself, retries with feedback, and **pauses for Asha's approval**:

![Diagram](../assets/diagrams/adce650ea63f.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  S([start]) --> P[parse] --> E[extract] --> V{validate}
  V -- problems, tries left --> R[retry] --> E
  V -- clean, or out of tries --> A[[admin_review ⏸]]
  A -- approved --> D[(store)]
  A -- rejected --> X([end])
```

</details>


| Day | You'll learn | You'll build |
|:-:|---|---|
| 1 | Workflows vs agents, the five patterns, ReAct, tool design | A ReAct loop by hand |
| 2 | `create_agent`, memory, structured responses, middleware | A CS201 rubric assistant |
| 3 | LangGraph: state, nodes, edges, reducers, supersteps | Your first `StateGraph`, with a loop |
| 4 | `Send` map-reduce, checkpoints, `interrupt()`, subgraphs, streaming | A parallel marker that pauses for Asha |
| 5 | 🧱 Project day | Exam understanding v2 |

## Part 2: What is an agent? Workflows vs agents

"Agent" is the most overloaded word in AI right now. Anthropic's essay *Building Effective Agents* gives the clearest split, and we'll use it all course. Both are **agentic systems**, but:

- A **workflow** is LLMs and tools orchestrated through **code paths you wrote in advance**. You decide the steps.
- An **agent** is an LLM that **dynamically directs its own process and tool use**, in a loop, until it decides it's done. The model decides the steps.

![Diagram](../assets/diagrams/6262547275ae.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  subgraph Workflow["Workflow: you draw the path"]
    direction LR
    a1[step 1: LLM] --> a2[step 2: code] --> a3[step 3: LLM]
  end
  subgraph Agent["Agent: the model picks the path"]
    direction LR
    b1[LLM] -- tool call --> b2[tools]
    b2 -- result --> b1
    b1 -- done --> b3([answer])
  end
```

</details>


🍳 **An analogy.** A workflow is a recipe: anyone following it gets the same dish. An agent is a chef with a stocked kitchen: more flexible, but you can't be sure exactly what they'll do, and it costs more.

| | Workflow | Agent |
|---|---|---|
| Predictable? | ✅ Yes, same steps every time | ⚠️ Less so |
| Cost and latency | Lower, and known in advance | Higher, and varies |
| Good for | Well-defined tasks you can break into steps | Open-ended tasks where the steps can't be predicted |
| Debugging | Easy: look at the step that failed | Harder: you read a trace |

💡 The golden rule from the essay: **find the simplest solution that works, and only add complexity when it clearly helps.** Often that's a single LLM call with good retrieval. Sometimes it's a workflow. Occasionally it's an agent.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Before reading on, decide: is GradeWise's exam extractor (parse → extract → validate → retry → approve) a <b>workflow</b> or an <b>agent</b>? What about the week 5 grader, which decides for itself when to look up evidence in the course notes and when to flag an answer for a human?</span>
        </td>
    </tr>
</table>

*Answer:* the extractor is a **workflow** (we drew every path), and the grader is an **agent** (it chooses its own tool calls). GradeWise uses both, and you'll see that's normal.

### The five patterns, in plain Python

Almost every agentic system is built from five patterns. To see their *shapes* without spending a cent, we'll use tiny Python functions as stand-ins for LLM calls. Our stand-in "marker" does keyword matching against the real rubric for Q3 (deadlocks):

In [ ]:
# 🎭 A stand-in for an LLM call: it marks by keyword matching against a criterion's keywords.
def keyword_mark(answer: str, question_id: str, strictness: int = 1) -> dict:
    """Award a criterion's marks if at least `strictness` of its keywords appear in the answer."""
    awarded, hits = 0.0, []
    for c in QUESTIONS[question_id].criteria:
        found = [k for k in c.keywords if k.lower() in answer.lower()]
        if len(found) >= strictness:
            awarded += c.marks
            hits.append(c.id)
    return {"question": question_id, "awarded": awarded, "max": QUESTIONS[question_id].max_marks, "criteria_met": hits}


sam_q3 = (
    "A deadlock is when processes are blocked, each waiting for a resource held by another. "
    "Conditions: mutual exclusion and circular wait."
)
print(QUESTIONS["Q3"].text)
keyword_mark(sam_q3, "Q3")

#### Pattern 1: Prompt chaining
Split a task into a fixed sequence of steps, each working on the previous step's output. You can add a **gate** (a plain code check) between steps to stop early.

![Diagram](../assets/diagrams/49003b658883.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  I([answer]) --> L1[LLM: mark] --> G{gate: valid?} -- yes --> L2[LLM: write feedback] --> O([feedback])
  G -- no --> F([fail fast])
```

</details>


In [ ]:
def write_feedback(result: dict) -> str:  # 🎭 stand-in for a second LLM call
    missing = [c.id for c in QUESTIONS[result["question"]].criteria if c.id not in result["criteria_met"]]
    if not missing:
        return f"Full marks ({result['awarded']:g}/{result['max']:g}). Excellent!"
    return f"{result['awarded']:g}/{result['max']:g}. Look again at: {', '.join(missing)}."


def chain(answer: str, question_id: str) -> str:
    result = keyword_mark(answer, question_id)  # step 1
    if result["awarded"] > result["max"]:  # 💡 the gate: plain code, no LLM needed
        raise ValueError("Marker awarded more than the maximum!")
    return write_feedback(result)  # step 2


print(chain(sam_q3, "Q3"))
print(chain("A deadlock is when the computer freezes.", "Q3"))

#### Pattern 2: Routing
Classify the input, then send it to a specialised handler. Each handler can have its own prompt, tools, or even a cheaper model.

![Diagram](../assets/diagrams/e9600054dd31.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  I([student message]) --> R{router} -- remark --> H1[re-mark handler]
  R -- concept --> H2[tutor handler]
  R -- admin --> H3[admin handler]
```

</details>


In [ ]:
def route(message: str) -> str:  # 🎭 in real life: a small, cheap LLM call with structured output
    text = message.lower()
    if "re-mark" in text or "remark" in text or "unfair" in text:
        return "remark"
    if "deadline" in text or "when" in text:
        return "admin"
    return "concept"


HANDLERS = {
    "remark": lambda m: "📝 Logged for Asha: re-mark requests always go to a human.",
    "admin": lambda m: "📅 Results are published one week after the exam.",
    "concept": lambda m: "🧑‍🏫 Tutor mode: let's look at the course notes together…",
}

for message in [
    "I think my Q3 mark is unfair, can I get a re-mark?",
    "When do results come out?",
    "What is circular wait?",
]:
    print(f"{route(message):8} ← {message}\n          {HANDLERS[route(message)](message)}")

#### Pattern 3: Parallelisation
Run independent LLM calls **at the same time**, then combine them. Two flavours:
- **Sectioning:** different sub-tasks in parallel (mark the answer *and* check it for prompt-injection, at once)
- **Voting:** the *same* task several times, then take the majority or median, for more confidence

![Diagram](../assets/diagrams/4968eec10bf2.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  I([answer]) --> M1[marker: lenient] & M2[marker: normal] & M3[marker: strict]
  M1 & M2 & M3 --> A[aggregate: median]
```

</details>


In [ ]:
# 💡 Voting: three "markers" with different strictness, run in parallel, then take the median.
with ThreadPoolExecutor() as pool:
    votes = list(pool.map(lambda s: keyword_mark(sam_q3, "Q3", strictness=s)["awarded"], [1, 2, 3]))

print("Votes:", votes, "→ median:", statistics.median(votes))
spread = max(votes) - min(votes)
print("⚠️ The markers disagree: flag for a human!" if spread >= 2 else "✅ The markers agree closely.")

#### Pattern 4: Orchestrator-workers
Like parallelisation, but the sub-tasks **aren't known in advance**: an orchestrator looks at the input, decides what work is needed, hands each piece to a worker, then a synthesiser combines the results. In LangGraph this is exactly what `Send` does (day 4).

![Diagram](../assets/diagrams/0d8c0cfd010f.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  I([answer sheet]) --> O[orchestrator: which questions are here?]
  O --> W1[worker: Q2a] & W2[worker: Q3] & W3[worker: …]
  W1 & W2 & W3 --> S[synthesiser: total]
```

</details>


In [ ]:
answer_sheet = """
Q2a: A process is a program in execution with its own address space and a PCB.
Q3: A deadlock is when processes are blocked waiting for a resource held by another. Mutual exclusion, hold and wait.
Q4a: When two threads access shared data and the result depends on the timing.
"""

# 🎭 The orchestrator decides the work at RUN time: however many answers this sheet has.
tasks = re.findall(r"^(Q\w+): (.+)$", answer_sheet, flags=re.MULTILINE)
print("Orchestrator found:", [qid for qid, _ in tasks])

with ThreadPoolExecutor() as pool:  # the workers
    results = list(pool.map(lambda t: keyword_mark(t[1], t[0]), tasks))

for r in results:  # the synthesiser
    print(f"  {r['question']:4} {r['awarded']:g}/{r['max']:g}")
print(f"Total: {sum(r['awarded'] for r in results):g}/{sum(r['max'] for r in results):g}")

#### Pattern 5: Evaluator-optimiser
One step **generates**, another **evaluates** against clear criteria and gives feedback, and the loop repeats until the evaluator is happy (or we run out of tries). This is the pattern behind GradeWise's validate → retry loop on Friday.

![Diagram](../assets/diagrams/ced875a7a0fb.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
flowchart LR
  G[generator] -- draft --> E{evaluator}
  E -- feedback --> G
  E -- accepted --> O([output])
```

</details>


In [ ]:
def generate_feedback(result: dict, critique: list[str]) -> str:  # 🎭 the "generator"
    text = write_feedback(result)
    if "be kind" in critique:
        text = "Good start! " + text
    if "name the topic" in critique:
        text += " (Hint: the four Coffman conditions.)"
    return text


def evaluate(feedback: str) -> list[str]:  # 🎭 the "evaluator": clear, checkable criteria
    problems = []
    if not feedback.startswith(("Good", "Full", "Excellent")):
        problems.append("be kind")
    if "Coffman" not in feedback and "Full marks" not in feedback:
        problems.append("name the topic")
    return problems


result, critique = keyword_mark("Deadlock means processes wait for a resource held by another.", "Q3"), []
for attempt in range(1, 4):  # ⚠️ ALWAYS cap the loop: an evaluator that is never happy would loop forever
    draft = generate_feedback(result, critique)
    critique = evaluate(draft)
    print(f"Attempt {attempt}: {draft}\n   evaluator says: {critique or 'accepted ✅'}")
    if not critique:
        break

#### And finally: the agent

An agent is the simplest diagram of all: a model in a loop with tools, deciding for itself what to do next. Here's how the patterns map onto GradeWise:

| Pattern | Where it shows up in GradeWise |
|---|---|
| Prompt chaining | Week 4: transcribe → map answers to questions → confidence check |
| Routing | Week 4: adaptive RAG decides whether to retrieve at all |
| Parallelisation | Week 5: grade all 15 questions at once |
| Orchestrator-workers | Week 5: `Send` one grading task per answer found on the sheet |
| Evaluator-optimiser | **This Friday:** validate the extracted exam, retry with feedback |
| Agent | Week 5: the grader agent with `retrieve_reference`, `get_rubric`, `flag_for_review` |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">Clients rarely ask for "an agent". They ask for something that's <b>reliable, affordable and explainable</b>. Being able to say "this part is a fixed workflow, so it costs the same every time and is easy to audit; only this one step is an agent, and it's capped at 5 model calls" is exactly the kind of design reasoning that wins trust, and that schools, insurers and banks will insist on.</span>
        </td>
    </tr>
</table>

## Part 3: The ReAct loop

**ReAct** (*Reason + Act*) is the loop inside nearly every tool-using agent:

1. 🧠 **Reason:** the model reads the conversation and decides what to do
2. 🛠️ **Act:** it asks for a tool call (a name + arguments as JSON)
3. 👀 **Observe:** *your code* runs the tool and appends the result as a `ToolMessage`
4. 🔁 Repeat, until the model answers without asking for a tool

![Diagram](../assets/diagrams/e585c680124d.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
sequenceDiagram
  participant U as Asha
  participant M as Model
  participant T as Tools
  U->>M: "How many marks is Q3, and what earns them?"
  M->>T: get_rubric(question_id="Q3")
  T-->>M: Q3 rubric (4 marks, 3 criteria)
  M->>U: "Q3 is worth 4 marks: 2 for the definition, …"
```

</details>


You met `@tool` and `bind_tools` in week 1 day 4. Here's a tool over Asha's real rubric:

In [ ]:
@tool(parse_docstring=True)
def get_rubric(question_id: str) -> str:
    """Look up the marking rubric for one CS201 midterm question.

    Args:
        question_id: The question ID exactly as printed in the exam JSON, e.g. "Q3" or "Q2a".
    """
    q = QUESTIONS.get(question_id)
    if q is None:  # 💡 a helpful message the model can recover from, not an exception
        return f"No question {question_id!r}. Valid IDs: {', '.join(QUESTIONS)}"
    lines = [f"{q.id} ({q.max_marks:g} marks): {q.text}"]
    lines += [f"- {c.id} [{c.marks:g}]: {c.description}" for c in q.criteria]
    return "\n".join(lines)


tools = {t.name: t for t in [get_rubric]}
print(get_rubric.invoke({"question_id": "Q3"}))

Now let's write the loop ourselves. It's only a few lines, and once you've seen it, `create_agent` tomorrow will hold no mysteries. To run it offline, we use a **🎭 stand-in model** that replays a script of messages. It never reads the prompt: it just shows you the exact messages a real model would send back.

In [ ]:
# 🎭 A STAND-IN model, so you can watch agents work with no API key and no cost.
# 💡 GenericFakeChatModel replies with the messages we script, in order. It can't really call
#    tools, so bind_tools() just returns the same model: the tool calls are in our script.
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage


class ScriptedModel(GenericFakeChatModel):
    """Replays a script of AIMessages. ⚠️ A teaching prop, not an LLM: it never reads your prompt."""

    def bind_tools(self, tools, **kwargs):
        return self


def scripted(*replies: AIMessage) -> ScriptedModel:
    return ScriptedModel(messages=iter(replies))


def call(name: str, call_id: str = "call_1", **args) -> AIMessage:
    """A scripted AIMessage that asks for ONE tool call, exactly like a real model would."""
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": call_id}])

In [ ]:
def react(model, question: str, max_steps: int = 5) -> str:
    """The ReAct loop by hand: call the model, run any tools it asks for, repeat."""
    messages = [
        SystemMessage("You are GradeWise, a CS201 marking assistant. Use tools; never guess marks."),
        HumanMessage(question),
    ]
    for step in range(1, max_steps + 1):  # ⚠️ a step cap: agents must never loop forever
        ai = model.invoke(messages)  # 🧠 Reason
        messages.append(ai)
        if not ai.tool_calls:  # no tool requested: this is the final answer
            print(f"Step {step} 💬 final answer")
            return ai.content
        for tc in ai.tool_calls:  # 🛠️ Act
            print(f"Step {step} 🛠️  {tc['name']}({tc['args']})")
            observation = tools[tc["name"]].invoke(tc["args"])  # 👀 Observe
            messages.append(ToolMessage(observation, tool_call_id=tc["id"]))
    return "Stopped: too many steps."


standin = scripted(
    call("get_rubric", question_id="Q3"),
    AIMessage(content="Q3 is worth 4 marks: 2 for defining deadlock, and 1 each for naming two Coffman conditions."),
)
print(react(standin, "How many marks is Q3 worth, and what earns them?"))

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">Before running the next cell with a <b>real</b> model, predict: how many times will it call the model if you ask about Q3 <i>and</i> Q9 in one question? A good model may ask for <b>both</b> rubrics in a single step (parallel tool calls), so it might be only 2 model calls, not 3.</span>
        </td>
    </tr>
</table>

### 🔑 The same loop, with a real model
Exactly the same `react()` function. The only change: a real model that has been told about our tool with `bind_tools`.

In [ ]:
from gradewise.models import chat_model

llm = chat_model(temperature=0).bind_tools([get_rubric])
display(Markdown(react(llm, "How many marks are Q3 and Q9 worth together, and what earns them?")))

## Part 4: Designing great tools

A model only knows your tool through three things: its **name**, its **description** (the docstring) and its **argument schema**. That's the *agent-computer interface*, and Anthropic's advice is to put as much care into it as you would into a human UI. Let's see what the model actually receives. (💡 `@tool(parse_docstring=True)` moved the `Args:` line of our docstring into the schema, as the argument's own description.)

In [ ]:
print(json.dumps(convert_to_openai_tool(get_rubric), indent=2))

Now compare a badly designed tool. It works, technically:

In [ ]:
@tool
def lookup(x: str, y: int = 0) -> dict:
    """Gets data."""
    return QUESTIONS[x].model_dump()  # ⚠️ crashes on a bad ID, and dumps ~1,000 tokens of JSON


print(json.dumps(convert_to_openai_tool(lookup)["function"], indent=2))
try:
    lookup.invoke({"x": "q3"})
except KeyError as err:
    print("💥 KeyError:", err, "← the agent gets no hint about what went wrong")
print("✅ good tool:", get_rubric.invoke({"question_id": "q3"}))

### 🛠️ The checklist for a great tool
| Do | Why |
|---|---|
| A **specific name**: `get_rubric`, not `lookup` | The model picks tools by name first |
| A **docstring that says when to use it**, with an example argument | The docstring *is* the prompt for the tool |
| **Typed, named arguments** (`question_id: str`, `Literal[...]` for fixed choices) | They become a JSON schema the model must follow |
| **Return concise text**, only what the model needs | Every returned token costs money and attention |
| **Return helpful errors** ("Valid IDs: …") instead of raising | The model can read the hint and correct itself |
| **Few, focused tools** that don't overlap | Two tools that do the same thing confuse models |
| **Make risky tools ask first** (e.g. `save_marks`) | Day 2 middleware and day 4 `interrupt()` |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;"><b>Tools run your code, with your permissions.</b> The model only <i>asks</i>; your code executes. Never give an agent a tool you wouldn't give a stranger: no raw SQL, no shell, no "write any file". Give it <code>get_rubric(question_id)</code>, not <code>run_query(sql)</code>. And anything that changes data should need a human's approval.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">The exam extractor you'll finish on Friday is a <b>workflow</b> (evaluator-optimiser plus a human approval step): open <code>gradewise/exam/graph.py</code> and spot the patterns. The week 5 grader is an <b>agent</b> whose tools (<code>retrieve_reference</code>, <code>get_rubric</code>, <code>flag_for_review</code>) follow today's checklist.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- 🗺️ previewed the week, and the exam workflow you'll ship on Friday
- 🧠 learned the difference between a **workflow** (you draw the path) and an **agent** (the model picks it)
- 🧩 ran all five patterns in plain Python: chaining, routing, parallelisation, orchestrator-workers, evaluator-optimiser
- 🔁 wrote the **ReAct loop** by hand, with a stand-in model and (🔑) a real one
- 🛠️ saw what a model actually "sees" of a tool, and why `get_rubric` beats `lookup`

**Tomorrow (Day 2):** LangChain's `create_agent` builds that ReAct loop for you, with memory, structured responses and **middleware** for logging, limits, PII and human approval.

⏭ **Next:** `week3/day2.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Redesign a bad tool. Below is the tool a hurried colleague wrote: <code>def marks(x, y): return QUESTIONS[x].criteria[y].marks</code>. Turn it into a great tool called <code>get_criterion_marks</code>: a clear docstring with an example, typed arguments (a question ID and a criterion ID like <code>"Q3.c2"</code>), and helpful error messages for an unknown question <i>and</i> an unknown criterion. Then run it through the hand-made <code>react()</code> loop with a scripted stand-in. Solution: <code>week3/solutions/day1_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• Anthropic, <i>Building Effective Agents</i>: <a href="https://www.anthropic.com/engineering/building-effective-agents">anthropic.com/engineering/building-effective-agents</a><br/> • Anthropic, <i>Writing effective tools for agents</i>: <a href="https://www.anthropic.com/engineering/writing-tools-for-agents">anthropic.com/engineering/writing-tools-for-agents</a><br/> • The ReAct paper (Yao et al., 2022): <a href="https://arxiv.org/abs/2210.03629">arxiv.org/abs/2210.03629</a><br/> • LangGraph, <i>Workflows and agents</i>: <a href="https://docs.langchain.com/oss/python/langgraph/workflows-agents">docs.langchain.com/oss/python/langgraph/workflows-agents</a></span>
        </td>
    </tr>
</table>